# Agent 第 03 课：手写一个真正的 ReAct Agent（Hermes 格式）

前两课我们搭好了骨架 + 定好了协议。这一课把它们**接起来**，做一个**真正能干活**的 agent：

- 多步循环：Thought → Tool Call → Tool Response → Thought → ... → Final Answer
- 支持多个工具
- 有保险丝（max_steps）
- 有完整 trace（每一步都能打印，便于调试）

这是整个课程里**代码最重要的一节**。之后所有进阶话题（memory、planning、reflection、multi-agent）都建立在这一课的 agent loop 之上。

## 1. Agent Loop 的一张大图

```
  ┌────────────────────────────────────────────────────┐
  │  messages = [system, user]                         │
  └────────────────────────────────────────────────────┘
                 │
                 ▼
  ┌──── for step in range(max_steps): ────┐
  │  resp = LLM(messages)                  │
  │  append assistant resp to messages     │
  │                                        │
  │  if <tool_call> in resp:               │
  │      result = exec(tool, args)         │
  │      append <tool_response> as 'user'  │
  │      continue                          │
  │  elif 'Final Answer:' in resp:         │
  │      return extract(resp)              │
  │  else:                                 │
  │      # 模型走神了，给它一句提醒         │
  │      append hint, continue             │
  └────────────────────────────────────────┘
```

注意两个关键细节：

1. **`<tool_response>` 是以 `role='user'` 塞回去的**——对 LLM 来说"工具结果"就是环境给它的新输入，它不知道自己调过工具，只看到上下文里有一段 tool_call 和一段 tool_response。
2. **每一步都要保留 assistant 的原始输出**（包括那些 `<tool_call>` 标签）。不能只塞工具结果，不然模型会忘记自己为什么调这个工具。

In [ ]:
import json, re, math, datetime
from openai import OpenAI

client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
chat_model = next((m for m in ['qwen/qwen3.6-35b-a3b','qwen3.6-35b-a3b','qwen3.5-35b-a3b'] if m in model_ids), model_ids[0])
print('chat =', chat_model)

## 2. 工具集

三件套：计算器、查今天日期、算两个日期差。都是"LLM 靠直觉做容易错"的任务。

In [ ]:
def tool_calculator(expr: str):
    allowed = {k: getattr(math, k) for k in ['sqrt','pi','e','sin','cos','log']}
    return eval(expr, {'__builtins__': {}}, allowed)

def tool_today():
    return datetime.date.today().isoformat()

def tool_days_between(date1: str, date2: str):
    d1 = datetime.date.fromisoformat(date1)
    d2 = datetime.date.fromisoformat(date2)
    return abs((d2 - d1).days)

TOOLS = [
    {'name': 'calculator',
     'description': 'Evaluate a math expression like "3*(4+5)" or "sqrt(144)". Supports sqrt, sin, cos, log, pi, e.',
     'parameters': {'type':'object', 'properties': {'expr':{'type':'string'}}, 'required':['expr']}},
    {'name': 'today',
     'description': "Returns today's date in ISO format (YYYY-MM-DD). Takes no arguments.",
     'parameters': {'type':'object','properties':{},'required':[]}},
    {'name': 'days_between',
     'description': 'Computes the number of days between two ISO dates.',
     'parameters': {'type':'object',
                    'properties': {'date1':{'type':'string'},'date2':{'type':'string'}},
                    'required':['date1','date2']}},
]

TOOL_IMPL = {
    'calculator': lambda a: tool_calculator(**a),
    'today':      lambda a: tool_today(),
    'days_between': lambda a: tool_days_between(**a),
}

## 3. 系统 prompt

三个关键指令：
1. 告诉它工具 schema
2. 告诉它 Hermes 格式
3. **告诉它什么时候停**（用 `Final Answer:` 开头）

In [ ]:
def build_system_prompt(tools):
    return f"""You are a careful assistant. You can call tools to get facts you are not sure about.

Available tools (JSON schema):
{json.dumps(tools, indent=2)}

Protocol:
- When you need a tool, output ONLY one tool call in this exact format:
  <tool_call>
  {{"name": "<tool_name>", "arguments": {{ ... }}}}
  </tool_call>
  Then stop and wait for the <tool_response>.
- You may call tools multiple times across turns.
- When you have enough information, reply with plain text starting exactly with:
  Final Answer: <your answer>
- Do not fabricate tool outputs. Do not call tools that are not listed.
- Keep thoughts short."""

SYSTEM = build_system_prompt(TOOLS)

## 4. 解析器

两条模式——`<tool_call>` 或 `Final Answer:`。写得宽容一点，模型偶尔会加空格、加代码块。

In [ ]:
TOOL_CALL_RE = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.S)
FINAL_RE = re.compile(r'Final Answer:\s*(.+)', re.S)

def parse_llm_output(text):
    m = TOOL_CALL_RE.search(text)
    if m:
        try:
            call = json.loads(m.group(1))
            return ('tool_call', call)
        except json.JSONDecodeError as e:
            return ('error', f'bad JSON in tool_call: {e}')
    m = FINAL_RE.search(text)
    if m:
        return ('final', m.group(1).strip())
    return ('other', text)

## 5. Agent 主循环

**这就是 agent 的核心**。看熟它。

In [ ]:
def run_agent(user_question, max_steps=8, verbose=True):
    messages = [{'role':'system','content':SYSTEM},
                {'role':'user','content':user_question}]
    for step in range(1, max_steps+1):
        resp = client.chat.completions.create(
            model=chat_model, temperature=0, messages=messages)
        text = resp.choices[0].message.content
        messages.append({'role':'assistant','content':text})
        kind, payload = parse_llm_output(text)
        if verbose:
            print(f'--- step {step} [{kind}] ---')
            print(text)
            print()
        if kind == 'final':
            return payload, messages
        if kind == 'tool_call':
            name = payload.get('name')
            args = payload.get('arguments', {})
            if name not in TOOL_IMPL:
                obs = {'error': f'unknown tool {name}'}
            else:
                try:
                    obs = {'result': TOOL_IMPL[name](args)}
                except Exception as e:
                    obs = {'error': f'{type(e).__name__}: {e}'}
            tool_resp = f'<tool_response>\n{json.dumps({"name": name, "content": obs})}\n</tool_response>'
            messages.append({'role':'user','content':tool_resp})
            if verbose: print('>>> tool result:', obs, '\n')
            continue
        if kind == 'error':
            messages.append({'role':'user','content':f'Your last tool_call had an error: {payload}. Please retry with a valid JSON tool_call or give a Final Answer.'})
            continue
        # 走神了
        messages.append({'role':'user','content':'Please either issue a <tool_call>...</tool_call> or give a Final Answer: ...'})
    return None, messages

## 6. 跑一个需要多步的问题

"**从今天到 2030 年元旦还有多少天？用天数乘以 24 得到总小时数。**"

这个任务强迫 agent：
1. 调 `today` 拿当前日期
2. 调 `days_between` 算天数差
3. 调 `calculator` 乘 24
4. 出最终答案

In [ ]:
answer, trace = run_agent('How many hours are there from today until 2030-01-01? Multiply days by 24.')
print('=== FINAL ANSWER ===')
print(answer)

## 7. 再跑一个边界情况：模型本可以直接答的问题

"**法国的首都是哪里？**" —— 不需要任何工具，模型自己就知道。

好的 agent 应该**跳过工具直接出 Final Answer**。如果它也去调工具，说明 prompt 还不够好（或者模型太弱）。

In [ ]:
answer, _ = run_agent('What is the capital of France?')
print('=== FINAL ANSWER ===')
print(answer)

## 8. 工程直觉

1. **Agent 循环非常简单**。本课 50 行代码就搞定了 ReAct agent 的骨架。**框架不是魔法，它们内部也就是这样**。LangChain 的 `AgentExecutor` 去掉日志和 callback 之后，核心和上面这段 `run_agent` 几乎一样。
2. **工具结果必须以 user role 塞回**。很多人一开始塞成 assistant/system 都不对——对 LLM 来说工具是"环境"，环境说话的 role 是 user。
3. **保留 assistant 原始输出**。不要只塞工具结果，模型需要看到"我刚刚说过我要调这个工具"才能连贯推理。
4. **三种模型异常必须处理**：未知工具名、工具执行异常、模型走神（既不调工具也不给答案）。三种都要"告诉模型怎么做才对"，而不是直接崩。
5. **max_steps 不只是保险丝**。它还是**成本上限**——一个失控的 agent 一分钟能烧掉几十块钱。本地模型不心疼，线上用 Bedrock 或 OpenAI 时这一句代码可能救你的工资。
6. **verbose trace 是 agent 开发的生命线**。不要等部署了再加日志。第一天就打印每一步。

---

下一课：**第 04 课 多工具路由 + 工具设计心法**——工具多了之后，模型选错、参数错、重复调用的问题会浮现。我们讲怎么设计工具集，让 agent 不犯这些错。